# eSASRec 2×2 reproduction

Controlled primary-paper experiment on **Amazon Beauty, full-catalog temporal evaluation**:

| Architecture | Full CE | Sampled Softmax 256 |
|---|---|---|
| SASRec | SASRec+FullCE | SASRec+SS |
| LiGR/SwiGLU | LiGR+FullCE | **eSASRec** |

All four cells use the released eSASRec Beauty geometry: `d=64`, `1 block`, `1 head`, `dropout=.2`, `FF mult=4`, `max_len=50`, `lr=1e-3`. The run checkpoints to Drive and is resumable.

This is intentionally separate from the authors’ academic sampled-ranking protocol; our primary outcome is full-catalog NDCG@10 under the Sparse Walker paper protocol.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

!rm -rf /content/Sparsewalker
!git clone -q https://github.com/hanialshater/Sparsewalker-.git /content/Sparsewalker
%cd /content/Sparsewalker
!pip -q install -e .
!python -m pytest -q tests/test_esasrec.py

import torch
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

## Run Beauty 2×2

The default is intentionally paper-like on optimization but conservative on runtime: 200 epoch cap, patience 50, validation every 5 epochs, batch 256. Each model writes `last.pt`, `best.pt`, `history.csv`, and `done.json`. Re-running the cell resumes incomplete models and skips completed ones.

In [ ]:
OUT='/content/drive/MyDrive/sparsewalker_esasrec_2x2'
!python benchmarks/run_esasrec_2x2.py \
  --dataset beauty \
  --seed 42 \
  --batch-size 256 \
  --eval-batch-size 1024 \
  --max-epochs 200 \
  --patience 50 \
  --eval-every 5 \
  --lr 1e-3 \
  --weight-decay 0 \
  --n-negs 256 \
  --ss-chunk-size 2048 \
  --output-dir "$OUT"

## Inspect decomposition

The most useful quantities are:

- `SASRec_SS_minus_FullCE`: effect of sampled softmax on vanilla SASRec.
- `LiGR_FullCE_minus_SASRec_FullCE`: architecture-only effect.
- `eSASRec_minus_SASRec_SS`: LiGR effect when both use SS.
- `interaction_LiGR_x_SS`: whether the two modifications are additive or interact.

In [ ]:
import json, pandas as pd
from pathlib import Path
root=Path(OUT)/'beauty'/'seed42'
summary=pd.read_csv(root/'summary.csv')
display(summary.sort_values('NDCG@10', ascending=False))
print(json.dumps(json.loads((root/'decomposition.json').read_text()), indent=2))

## Optional next sanity: ML-1M

Only run this after Beauty finishes. The authors’ academic benchmark reports a clearer eSASRec lift on ML-1M, so this is our literature sanity check. The script automatically switches to `d=64`, `2 blocks`, `1 head`, `dropout=.1`, `max_len=200`, 100 epochs.

In [ ]:
# Uncomment after Beauty finishes.
# !python benchmarks/run_esasrec_2x2.py \
#   --dataset ml1m --seed 42 --batch-size 128 --eval-batch-size 1024 \
#   --max-epochs 100 --patience 50 --eval-every 5 --lr 1e-3 \
#   --weight-decay 0 --n-negs 256 --ss-chunk-size 2048 --output-dir "$OUT"